# Tutorial 10: Disambiguation — Recovering Tokens from HLLSet Fingerprints

This tutorial introduces the `disambiguation` module, which recovers candidate tokens from HLLSet fingerprints using triangulation.

## What You'll Learn

1. **Token Entry Structure** — Hash decomposition into (reg, zeros, layer)
2. **Triangulation Tensor** — Sparse 4D storage for candidate lookup
3. **Disambiguation Engine** — Ingest, triangulate, and recover sequences
4. **De Bruijn Integration** — Using n-gram layers for sequence reconstruction

## The Challenge

An HLLSet contains only **fingerprints** (register indices + leading zero counts), NOT the original tokens.

Given an HLLSet, how do we recover WHICH tokens produced those fingerprints?

## The Solution: Triangulation

1. **Build a corpus** — Ingest known tokens with their n-gram contexts
2. **Index by fingerprint** — Store tokens at (layer, reg, zeros) coordinates
3. **Triangulate** — Use multiple n-gram layers to narrow candidates
4. **Reconstruct** — Apply De Bruijn to recover sequence

In [9]:
# Import modules
import sys
sys.path.insert(0, '..')

from core.disambiguation import (
    TokenEntry,
    TriangulationTensor,
    DisambiguationEngine,
    ParallelDisambiguator,
    START_TOKEN,
    END_TOKEN,
)
from core.hllset import HLLSet, murmur_hash64a

# HLL constants (default precision bits = 10)
P_BITS = 10
NUM_REGISTERS = 1 << P_BITS  # 1024 registers

## 1. Hash Decomposition

Every token hash is decomposed into three components.

In [10]:
# Hash a token using murmur_hash64a
token = "hello"
h = murmur_hash64a(token.encode())

print(f"Token: '{token}'")
print(f"Hash (64-bit): {h:#018x}")
print(f"Binary (first 32 bits): {h >> 32:032b}")

Token: 'hello'
Hash (64-bit): 0x1e68d17c457bf117
Binary (first 32 bits): 00011110011010001101000101111100


In [11]:
# Decompose the hash manually (mimicking HLL decomposition)
p_bits = 10  # HLL precision bits
reg = h & ((1 << p_bits) - 1)  # Lower p bits -> register index
zeros = (h >> p_bits).bit_length()  # Leading zeros approximation
layer = (h >> 56) & 0x7  # Top bits for layer assignment

print(f"\nHash decomposition:")
print(f"  reg (register index): {reg}  (range: 0-{NUM_REGISTERS-1})")
print(f"  zeros (leading zeros): {zeros}")
print(f"  layer (n-gram layer): {layer}  (range: 0-7)")


Hash decomposition:
  reg (register index): 279  (range: 0-1023)
  zeros (leading zeros): 51
  layer (n-gram layer): 6  (range: 0-7)


In [12]:
# Compare decompositions for different tokens
tokens = ["hello", "world", "python", "hllset"]
p_bits = 10

print(f"Hash decompositions:")
print(f"{'Token':<10} {'Hash':<18} {'Reg':>5} {'Zeros':>6} {'Layer':>6}")
print("-" * 50)
for t in tokens:
    h = murmur_hash64a(t.encode())
    reg = h & ((1 << p_bits) - 1)
    zeros = (h >> p_bits).bit_length()
    layer = (h >> 56) & 0x7
    print(f"{t:<10} {h:#018x} {reg:>5} {zeros:>6} {layer:>6}")

Hash decompositions:
Token      Hash                 Reg  Zeros  Layer
--------------------------------------------------
hello      0x1e68d17c457bf117   279     51      6
world      0x4d46ae3bbc0fb0f7   247     53      5
python     0x6355b9fc4247584b    75     53      3
hllset     0x4c010a1bd0b907ae   942     53      4


## 2. Token Entry

A `TokenEntry` wraps a token with its hash and decomposition.

In [13]:
# Create TokenEntry using the actual API
entry = TokenEntry.from_ntoken(("hello",))

print(f"TokenEntry:")
print(f"  token: {entry.token}")
print(f"  hash_full: {entry.hash_full:#018x}")
print(f"  hash_mod: {entry.hash_mod}")
print(f"  reg: {entry.reg}")
print(f"  zeros: {entry.zeros}")
print(f"  layer: {entry.layer}")

TokenEntry:
  token: ('hello',)
  hash_full: 0xd417125ccb971887
  hash_mod: 6279
  reg: 135
  zeros: 1
  layer: 0


In [14]:
# TokenEntry coordinates for tensor indexing
print(f"\nTensor coordinates: (layer={entry.layer}, reg={entry.reg}, zeros={entry.zeros})")


Tensor coordinates: (layer=0, reg=135, zeros=1)


## 3. Triangulation Tensor

A sparse 4D structure: `tensor[layer, reg, zeros, position]` → candidates

In [16]:
# Create tensor (uses default p_bits=10, which gives 1024 registers)
tensor = TriangulationTensor(p_bits=P_BITS)

print(f"TriangulationTensor:")
print(f"  p_bits: {tensor.p_bits}")
print(f"  num_registers: {tensor.num_registers}")

TriangulationTensor:
  p_bits: 10
  num_registers: 1024


In [17]:
# Add entries using add_entry method
entry1 = TokenEntry.from_ntoken(("apple",))
entry2 = TokenEntry.from_ntoken(("apple", "pie"))  # Bigram

tensor.add_entry(entry1)
tensor.add_entry(entry2)

print(f"After adding entries:")
print(f"  Stats: {tensor.stats()}")

After adding entries:
  Stats: {'total_entries': 2, 'unique_positions': 2, 'entries_by_layer': [1, 1, 0], 'positions_with_collisions': 0, 'max_collision_depth': 0, 'avg_collision_depth': 0}


In [18]:
# Lookup candidates at entry1's position
candidates = tensor.lookup(layer=entry1.layer, reg=entry1.reg, zeros=entry1.zeros)

print(f"\\nLookup at (layer={entry1.layer}, reg={entry1.reg}, zeros={entry1.zeros}):")
for c in candidates:
    print(f"  token: {c.token}")

\nLookup at (layer=0, reg=385, zeros=1):
  token: ('apple',)


## 4. Disambiguation Engine

The main interface for token recovery.

In [20]:
# Create engine (uses p_bits for register count)
engine = DisambiguationEngine(p_bits=P_BITS)

print(f"DisambiguationEngine:")
print(f"  p_bits: {engine.p_bits}")
print(f"  tensor stats: {engine.tensor.stats()}")

DisambiguationEngine:
  p_bits: 10
  tensor stats: {'total_entries': 0, 'unique_positions': 0, 'entries_by_layer': [0, 0, 0], 'positions_with_collisions': 0, 'max_collision_depth': 0, 'avg_collision_depth': 0}


In [22]:
# Ingest a corpus of token sequences
sequences = [
    ["the", "quick", "brown", "fox"],
    ["the", "lazy", "dog"],
    ["quick", "brown", "fox", "jumps"],
    ["brown", "fox", "jumps", "over"],
]

for seq in sequences:
    engine.ingest_tokens(seq)

print(f"After ingestion:")
print(f"  entries: {len(engine)}")
print(f"  stats: {engine.stats()}")

After ingestion:
  entries: 42
  stats: {'total_entries': 42, 'unique_positions': 42, 'entries_by_layer': [10, 15, 17], 'positions_with_collisions': 0, 'max_collision_depth': 0, 'avg_collision_depth': 0, 'p_bits': 10, 'layer_cardinalities': [10, 14, 19]}


In [23]:
# Build an HLLSet from a known sequence
test_sequence = ["the", "quick", "brown", "fox"]
test_hll = HLLSet.from_batch(test_sequence)

print(f"Test HLLSet:")
print(f"  tokens: {test_sequence}")
print(f"  |H| ≈ {test_hll.cardinality():.0f}")

Test HLLSet:
  tokens: ['the', 'quick', 'brown', 'fox']
  |H| ≈ 5


In [24]:
# Disambiguate - recover candidate tokens
candidates = engine.disambiguate_hllset(test_hll)

print(f"\nRecovered candidates:")
print(f"  {candidates}")
print(f"\nOriginal: {test_sequence}")


Recovered candidates:
  [DisambiguationResult(reg=573, zeros=0, candidates=[TokenEntry(token=('brown',), hash_full=14134106501628180029, hash_mod=13885, reg=573, zeros=0, layer=0, ref_hash=14134106501628180029, first_token='brown')], confidence=1.0, method='exact'), DisambiguationResult(reg=678, zeros=3, candidates=[TokenEntry(token=('fox',), hash_full=9807541256886067878, hash_mod=41638, reg=678, zeros=3, layer=0, ref_hash=9807541256886067878, first_token='fox')], confidence=1.0, method='exact'), DisambiguationResult(reg=892, zeros=2, candidates=[TokenEntry(token=('quick',), hash_full=2954666003754177404, hash_mod=54140, reg=892, zeros=2, layer=0, ref_hash=2954666003754177404, first_token='quick')], confidence=1.0, method='exact'), DisambiguationResult(reg=960, zeros=1, candidates=[TokenEntry(token=('the',), hash_full=4396381385683147712, hash_mod=39872, reg=960, zeros=1, layer=0, ref_hash=4396381385683147712, first_token='the')], confidence=1.0, method='exact')]

Original: ['the', '

## 5. N-gram Layers for Triangulation

Multiple n-gram layers provide cross-validation.

In [25]:
# Explain n-gram indexing
print(f"N-gram layer structure:")
print(f"  Layer 0: Unigrams (single tokens)")
print(f"  Layer 1: Bigrams (2-token sequences)")
print(f"  Layer 2: Trigrams (3-token sequences)")
print(f"\nExample for 'the quick brown':")
print(f"  Layer 0: 'the', 'quick', 'brown'")
print(f"  Layer 1: 'the_quick', 'quick_brown'")
print(f"  Layer 2: 'the_quick_brown'")

N-gram layer structure:
  Layer 0: Unigrams (single tokens)
  Layer 1: Bigrams (2-token sequences)
  Layer 2: Trigrams (3-token sequences)

Example for 'the quick brown':
  Layer 0: 'the', 'quick', 'brown'
  Layer 1: 'the_quick', 'quick_brown'
  Layer 2: 'the_quick_brown'


In [27]:
# Examine layer statistics (NUM_LAYERS=3: unigram, bigram, trigram)
stats = engine.stats()
print(f"Layer statistics:")
print(f"  Entries by layer: {stats['entries_by_layer']}")
print(f"  Layer cardinalities: {stats['layer_cardinalities']}")

Layer statistics:
  Entries by layer: [10, 15, 17]
  Layer cardinalities: [10, 14, 19]


## 6. START and END Markers

Special tokens mark sequence boundaries for De Bruijn reconstruction.

In [28]:
# Boundary markers
print(f"Boundary markers:")
print(f"  START_TOKEN: '{START_TOKEN}'")
print(f"  END_TOKEN: '{END_TOKEN}'")

# Show how they're used in n-grams
print(f"\nFor sequence ['a', 'b', 'c']:")
print(f"  Bigrams: ['{START_TOKEN}_a', 'a_b', 'b_c', 'c_{END_TOKEN}']")

Boundary markers:
  START_TOKEN: '<S>'
  END_TOKEN: '</S>'

For sequence ['a', 'b', 'c']:
  Bigrams: ['<S>_a', 'a_b', 'b_c', 'c_</S>']


## 7. Parallel Disambiguation

Use register-parallel processing for large HLLSets.

In [30]:
# Create parallel disambiguator from engine
parallel = ParallelDisambiguator.from_engine(engine)

print(f"ParallelDisambiguator:")
print(f"  num_registers: {parallel.num_registers}")
print(f"  num_layers: {parallel.num_layers}")

ParallelDisambiguator:
  num_registers: 1024
  num_layers: 3


In [33]:
# Disambiguate all registers in parallel
results = parallel.disambiguate_parallel(max_workers=4)

print(f"Parallel disambiguation results:")
print(f"  Registers processed: {len(results)}")
for reg, result in list(results.items())[:3]:  # Show first 3
    print(f"  Register {reg}: {len(result.surviving_unigrams)} surviving unigrams")

Parallel disambiguation results:
  Registers processed: 10
  Register 678: 1 surviving unigrams
  Register 892: 1 surviving unigrams
  Register 648: 1 surviving unigrams


## 8. Confidence Scoring

Score candidates by how many layers confirm them.

In [35]:
# Get disambiguation results with confidence scores
scored = engine.disambiguate_hllset(test_hll)

print(f"Candidates with confidence scores:")
print(f"{'Position':<12} {'Best Token':<15} {'Confidence':>10} {'Method':<12}")
print("-" * 50)
for r in scored[:10]:  # Top 10
    best = r.best_token if r.best_token else "None"
    print(f"({r.reg},{r.zeros}){'':<4} {best:<15} {r.confidence:>10.3f} {r.method:<12}")

Candidates with confidence scores:
Position     Best Token      Confidence Method      
--------------------------------------------------
(573,0)     brown                1.000 exact       
(678,3)     fox                  1.000 exact       
(892,2)     quick                1.000 exact       
(960,1)     the                  1.000 exact       


## 9. Practical Example: Document Fingerprint Recovery

Recover document content from its HLLSet fingerprint.

In [37]:
# Build corpus from multiple documents
documents = [
    "machine learning algorithms process data efficiently",
    "deep learning networks learn representations automatically",
    "neural networks are powerful machine learning models",
    "data science combines statistics and machine learning",
]

doc_engine = DisambiguationEngine(p_bits=P_BITS)

for doc in documents:
    tokens = doc.split()
    doc_engine.ingest_tokens(tokens)

print(f"Corpus ingested:")
print(f"  documents: {len(documents)}")
print(f"  entries: {len(doc_engine)}")

Corpus ingested:
  documents: 4
  entries: 80


In [38]:
# Create fingerprint of a target document
target = "neural networks are powerful models"
target_tokens = target.split()
target_hll = HLLSet.from_batch(target_tokens)

print(f"Target document: '{target}'")
print(f"Target tokens: {target_tokens}")
print(f"HLLSet |H| ≈ {target_hll.cardinality():.0f}")

Target document: 'neural networks are powerful models'
Target tokens: ['neural', 'networks', 'are', 'powerful', 'models']
HLLSet |H| ≈ 5


In [40]:
# Recover candidates
recovered = doc_engine.disambiguate_hllset(target_hll)

print(f"\nRecovered candidates:")
# Extract best_token from each DisambiguationResult
recovered_tokens = [r.best_token for r in recovered if r.best_token]
print(f"  {recovered_tokens}")
print(f"\nTarget tokens: {target_tokens}")

# Calculate recovery accuracy
correct = set(target_tokens) & set(recovered_tokens)
precision = len(correct) / len(recovered_tokens) if recovered_tokens else 0
recall = len(correct) / len(target_tokens) if target_tokens else 0

print(f"\nRecovery metrics:")
print(f"  Precision: {precision:.1%}")
print(f"  Recall: {recall:.1%}")


Recovered candidates:
  ['powerful', 'networks', 'models', 'are', 'neural']

Target tokens: ['neural', 'networks', 'are', 'powerful', 'models']

Recovery metrics:
  Precision: 100.0%
  Recall: 100.0%


## 10. De Bruijn Sequence Reconstruction

Combine disambiguation with De Bruijn for full sequence recovery.

In [41]:
from core.debruijn import DeBruijnGraph

# Recover with sequence order
def recover_sequence(engine, hll, k=3):
    """Recover ordered sequence using De Bruijn reconstruction."""
    # Get candidates with positions
    candidates = engine.disambiguate_with_positions(hll)
    
    # Build De Bruijn graph from n-gram candidates
    graph = DeBruijnGraph(k=k)
    
    # Add n-grams from candidates
    ngram_layer = engine.tensor.get_ngrams(k, candidates)
    for ngram in ngram_layer:
        graph.add_kmer(ngram)
    
    # Find Eulerian path
    result = graph.find_eulerian_path()
    return result.sequence if result else candidates

# Note: This is a simplified illustration
# The actual implementation handles edge cases and scoring

## 11. Tensor Sparsity

The tensor is sparse — most (layer, reg, zeros) cells are empty.

In [43]:
# Analyze tensor sparsity using stats()
from core.disambiguation import NUM_LAYERS

tensor_stats = doc_engine.tensor.stats()
filled_positions = tensor_stats['unique_positions']
total_entries = tensor_stats['total_entries']

# Theoretical max positions: NUM_LAYERS * NUM_REGISTERS * max_zeros (32)
max_positions = NUM_LAYERS * NUM_REGISTERS * 32
sparsity = 1.0 - (filled_positions / max_positions)

print(f"Tensor sparsity analysis:")
print(f"  Total entries: {total_entries:,}")
print(f"  Unique positions: {filled_positions:,}")
print(f"  Max possible positions: {max_positions:,}")
print(f"  Sparsity: {sparsity:.4%}")

Tensor sparsity analysis:
  Total entries: 80
  Unique positions: 80
  Max possible positions: 98,304
  Sparsity: 99.9186%


## 12. Collision Handling

Multiple tokens can map to the same (reg, zeros) — collisions.

In [45]:
# Find collision groups by examining tensor data
# Access internal _data dict to find positions with multiple entries
collisions = {
    pos: entries 
    for pos, entries in doc_engine.tensor._data.items() 
    if len(entries) >= 2
}

print(f"Collision groups (2+ tokens at same cell):")
for (layer, zeros, reg), entries in list(collisions.items())[:5]:
    tokens = [e.token for e in entries]
    print(f"  layer={layer}, reg={reg}, zeros={zeros}: {tokens}")

print(f"\nTotal collision groups: {len(collisions)}")
print(f"Max collision depth: {tensor_stats['max_collision_depth']}")

Collision groups (2+ tokens at same cell):

Total collision groups: 0
Max collision depth: 0


In [46]:
# Triangulation resolves collisions using multiple layers
print(f"\nCollision resolution via triangulation:")
print(f"  1. Token appears in cell (layer=0, reg=X, zeros=Y)")
print(f"  2. Its bigram appears in (layer=1, reg=X', zeros=Y')")
print(f"  3. Its trigram appears in (layer=2, reg=X'', zeros=Y'')")
print(f"  → Intersection of candidates across layers reduces ambiguity")


Collision resolution via triangulation:
  1. Token appears in cell (layer=0, reg=X, zeros=Y)
  2. Its bigram appears in (layer=1, reg=X', zeros=Y')
  3. Its trigram appears in (layer=2, reg=X'', zeros=Y'')
  → Intersection of candidates across layers reduces ambiguity


## Summary

In this tutorial, you learned:

1. **Hash Decomposition** — Split 64-bit hash into (reg, zeros, layer)
2. **Triangulation Tensor** — Sparse 4D index for candidate lookup
3. **Multi-layer Triangulation** — Cross-validate using n-gram layers
4. **Parallel Processing** — Register-parallel disambiguation

### Key Components

| Component | Purpose |
|-----------|--------|
| `TokenEntry` | Wraps token with hash decomposition |
| `TriangulationTensor` | Sparse (layer, reg, zeros, pos) → candidates |
| `DisambiguationEngine` | Ingest corpus, recover tokens |
| `ParallelDisambiguator` | Multi-worker disambiguation |

### The Triangulation Process

1. **Ingest** — Build tensor from known token sequences
2. **Query** — Extract (reg, zeros) from target HLLSet registers
3. **Lookup** — Find candidates at each (layer, reg, zeros)
4. **Triangulate** — Intersect candidates across layers
5. **Reconstruct** — Use De Bruijn to recover sequence order

### Limitations

- Only tokens **in the corpus** can be recovered
- Collisions require multi-layer triangulation
- Sequence recovery depends on n-gram coverage

### Next Steps

- **Tutorial 11**: Evolution — Track HLLSet changes over time